In [7]:
import pandas as pd
import numpy as np
import math
import warnings
warnings.filterwarnings('ignore')

In [2]:
df_placar2 = pd.read_csv("vlr_placar2.csv")
df_placar2

,URL,Mapa,Time1,Mapas_T1,Score_T1,Time2,Mapas_T2,Score_T2,Vencedor
0,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,Haven,LOUD,NaN,7,Cloud9,NaN,13,Cloud9
1,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,Abyss,LOUD,NaN,11,Cloud9,NaN,13,Cloud9
2,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,AllMaps,LOUD,0.0,18,Cloud9,2.0,26,Cloud9
3,https://www.vlr.gg/596399/envy-vs-evil-geniuse...,Abyss,ENVY,NaN,9,Evil Geniuses,NaN,13,Evil Geniuses
4,https://www.vlr.gg/596399/envy-vs-evil-geniuse...,Corrode,ENVY,NaN,13,Evil Geniuses,NaN,10,ENVY
...,...,...,...,...,...,...,...,...,...
473,https://www.vlr.gg/428012/sentinels-vs-mibr-ch...,Lotus,Sentinels,NaN,13,MIBR,NaN,7,Sentinels
474,https://www.vlr.gg/428012/sentinels-vs-mibr-ch...,Abyss,Sentinels,NaN,13,MIBR,NaN,5,Sentinels
475,https://www.vlr.gg/428012/sentinels-vs-mibr-ch...,Pearl,Sentinels,NaN,19,MIBR,NaN,21,MIBR
476,https://www.vlr.gg/428012/sentinels-vs-mibr-ch...,Split,Sentinels,NaN,13,MIBR,NaN,7,Sentinels


In [3]:
df_stats2 = pd.read_csv("vlr_stats2.csv")
df_stats2

,URL,Mapa_Num,Mapa,Jogador,Time,Agente,R,ACS,K,D,A,+/-,KAST,ADR,HS%,FK,FD,+/-_FK_FD
0,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,Darker,LOUD,Viper,1.12,256,18,16,6,2,70,166,30,2,1,1
1,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,cauanzin,LOUD,Sova,0.98,209,15,13,3,2,75,152,26,0,1,-1
2,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,Virtyy,LOUD,Yoru,0.70,178,14,16,4,-2,55,102,35,2,3,-1
3,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,pANcada,LOUD,Omen,0.67,145,11,13,1,-2,65,99,27,0,3,-3
4,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,lukxo,LOUD,Killjoy,0.65,192,13,16,3,-3,60,125,35,4,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4845,https://www.vlr.gg/428012/sentinels-vs-mibr-ch...,4,Split,nzr,MIBR,Skye,1.19,272,19,16,6,3,75,180,30,4,1,3
4846,https://www.vlr.gg/428012/sentinels-vs-mibr-ch...,4,Split,xenom,MIBR,Astra,0.83,143,10,12,6,-2,70,101,24,0,0,0
4847,https://www.vlr.gg/428012/sentinels-vs-mibr-ch...,4,Split,artzin,MIBR,Raze,0.72,172,10,15,6,-5,70,119,17,3,4,-1
4848,https://www.vlr.gg/428012/sentinels-vs-mibr-ch...,4,Split,cortezia,MIBR,Viper,0.71,174,11,14,2,-3,60,129,23,1,0,1


In [4]:
df_merged = pd.read_csv("merged_status_placar.csv")

In [9]:
# ─── Placar por mapa individual (sem AllMaps) ─────────────────────────────────
df_placar_mapas = df_placar2[df_placar2['Mapa'] != 'AllMaps'].copy()

# ─── Placar AllMaps ───────────────────────────────────────────────────────────
df_placar_all = df_placar2[df_placar2['Mapa'] == 'AllMaps'].copy()

# ─── Merge principal: stats + placar do mapa específico ──────────────────────
df_merged = df_stats2.merge(
    df_placar_mapas[['URL', 'Mapa', 'Time1', 'Time2', 'Score_T1', 'Score_T2', 'Vencedor']],
    on=['URL', 'Mapa'],
    how='left'
)

# ─── Merge secundário: traz dados do AllMaps pela URL ────────────────────────
df_merged = df_merged.merge(
    df_placar_all[['URL', 'Time1', 'Time2', 'Mapas_T1', 'Mapas_T2',
                   'Score_T1', 'Score_T2', 'Vencedor']],
    on='URL',
    how='left',
    suffixes=('', '_all')
)

# ─── Resolve Time1/Time2 e Vencedor ──────────────────────────────────────────
df_merged['Time1']    = df_merged['Time1'].fillna(df_merged['Time1_all'])
df_merged['Time2']    = df_merged['Time2'].fillna(df_merged['Time2_all'])
df_merged['Vencedor'] = df_merged['Vencedor'].fillna(df_merged['Vencedor_all'])

# ─── Preenche Score do mapa com Score total quando for AllMaps ────────────────
df_merged['Score_T1'] = df_merged['Score_T1'].fillna(df_merged['Score_T1_all'])
df_merged['Score_T2'] = df_merged['Score_T2'].fillna(df_merged['Score_T2_all'])

# ─── Adversário ───────────────────────────────────────────────────────────────
df_merged['Adversario'] = df_merged.apply(
    lambda row: row['Time2'] if row['Time'] == row['Time1'] else row['Time1'],
    axis=1
)

# ─── Score do time e adversário (round do mapa ou total no AllMaps) ───────────
df_merged['Score_Time'] = df_merged.apply(
    lambda row: row['Score_T1'] if row['Time'] == row['Time1'] else row['Score_T2'],
    axis=1
)
df_merged['Score_Adv'] = df_merged.apply(
    lambda row: row['Score_T2'] if row['Time'] == row['Time1'] else row['Score_T1'],
    axis=1
)

# ─── Mapas ganhos na série (só faz sentido no AllMaps, NaN nos mapas) ─────────
df_merged['Mapas_Time'] = df_merged.apply(
    lambda row: row['Mapas_T1'] if row['Time'] == row['Time1'] else row['Mapas_T2'],
    axis=1
)
df_merged['Mapas_Adv'] = df_merged.apply(
    lambda row: row['Mapas_T2'] if row['Time'] == row['Time1'] else row['Mapas_T1'],
    axis=1
)

# ─── Limpa colunas auxiliares ─────────────────────────────────────────────────
df_merged = df_merged.drop(columns=[
    'Time1', 'Time2', 'Time1_all', 'Time2_all',
    'Score_T1', 'Score_T2',
    'Score_T1_all', 'Score_T2_all',
    'Mapas_T1', 'Mapas_T2',
    'Vencedor_all'
])

df_merged.to_csv("merged_status_placar.csv", index=False)
df_merged.head(14)

,URL,Mapa_Num,Mapa,Jogador,Time,Agente,R,ACS,K,D,...,HS%,FK,FD,+/-_FK_FD,Vencedor,Adversario,Score_Time,Score_Adv,Mapas_Time,Mapas_Adv
0,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,Darker,LOUD,Viper,1.12,256,18,16,...,30,2,1,1,Cloud9,Cloud9,7.0,13.0,0.0,2.0
1,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,cauanzin,LOUD,Sova,0.98,209,15,13,...,26,0,1,-1,Cloud9,Cloud9,7.0,13.0,0.0,2.0
2,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,Virtyy,LOUD,Yoru,0.70,178,14,16,...,35,2,3,-1,Cloud9,Cloud9,7.0,13.0,0.0,2.0
3,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,pANcada,LOUD,Omen,0.67,145,11,13,...,27,0,3,-3,Cloud9,Cloud9,7.0,13.0,0.0,2.0
4,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,lukxo,LOUD,Killjoy,0.65,192,13,16,...,35,4,4,0,Cloud9,Cloud9,7.0,13.0,0.0,2.0
5,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,OXY,C9,Neon,1.44,282,21,14,...,29,5,2,3,Cloud9,LOUD,13.0,7.0,2.0,0.0
6,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,penny,C9,Viper,1.23,203,15,12,...,23,3,2,1,Cloud9,LOUD,13.0,7.0,2.0,0.0
7,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,v1c,C9,Omen,1.04,198,14,13,...,26,1,2,-1,Cloud9,LOUD,13.0,7.0,2.0,0.0
8,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,Xeppaa,C9,Sova,1.01,185,13,16,...,21,0,0,0,Cloud9,LOUD,13.0,7.0,2.0,0.0
9,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,Zellsis,C9,Killjoy,0.82,166,11,16,...,20,3,2,1,Cloud9,LOUD,13.0,7.0,2.0,0.0


In [5]:
def time_atual():
    df = pd.read_csv("vlr_stats_2026.csv")
    
    tag = df_stats2["Time"].unique()
    
    times = {}
    
    for sigla in tag:
        jogadores = df[df["Time"] == sigla]["Jogador"].unique()
        times[sigla] = list(jogadores)
    
    return times


times = time_atual()

times

{'LOUD': ['Darker', 'cauanzin', 'Virtyy', 'pANcada', 'lukxo'],
 'C9': ['OXY', 'penny', 'v1c', 'Xeppaa', 'Zellsis', 'Demon1'],
 'G2': ['BABYBAY', 'trent', 'valyn', 'leaf', 'jawgemo'],
 '100T': ['Cryocells', 'bang', 'vora', 'Asuna', 'Timotino'],
 'ENVY': ['Rossy', 'Inspire', 'keznit', 'Eggsterr', 'P0PPIN', 'Demon1'],
 'EG': ['dgzin', 'bao', 'Okeanos', 'C0M', 'supamen'],
 'KRÜ': ['benG', 'mta', 'NagZ', 'infiltrator', 'Governor'],
 'FUR': ['koalanoob', 'artzin', 'nerve', 'eeiu', 'alym'],
 'LEV': ['PxS', 'Sato', 'kiNgg', 'blowz', 'spike', 'Neon'],
 'NRG': ['brawk', 'keiko', 'mada', 'Ethan', 'skuba'],
 'MIBR': ['aspas', 'Mazino', 'Verno', 'tex', 'zekken'],
 'SEN': ['johnqt', 'cortezia', 'Reduxx', 'Kyu', 'N4RRATE'],
 '2G': []}

# Calculo de Impacto

In [11]:
REF_MEDIANO = {
    'K': 770.0, 'D': 840.0, 'A': 294.0, '+/-': -26.0,
    'FK': 120.0, 'FD': 124.0, '+/-_FK_FD': -4.0,
    'R': 0.949, 'ACS': 190.8, 'KAST': 71.1, 'ADR': 124.8,
    'HS': 27.5, 'Score_Time': 548.0, 'Score_Adv': 630.0,
    'Mapas_Time': 48.0, 'Mapas_Adv': 56.0
}

PESOS = {
    'ACS': 0.106, 'R': 0.104, 'ADR': 0.097, 'HS': 0.095,
    'KAST': 0.093, '+/-': 0.085, 'A': 0.079, 'D': 0.076,
    'K': 0.071, '+/-_FK_FD': 0.056, 'FK': 0.052, 'FD': 0.050,
    'Mapa_Num': 0.034
}


def calcular_impacto(r, acs, kast, adr, hs, tot_k, tot_d, tot_fk, tot_fd, jogos):
    def clip(x, min_v=0.5, max_v=1.5):
        return max(min(x, max_v), min_v)

    kd_rate  = (tot_k - tot_d) / jogos if jogos > 0 else 0
    fkd_rate = (tot_fk - tot_fd) / jogos if jogos > 0 else 0

    scores = {
        'R':          clip(r    / REF_MEDIANO['R']),
        'ACS':        clip(acs  / REF_MEDIANO['ACS']),
        'ADR':        clip(adr  / REF_MEDIANO['ADR']),
        'KAST':       clip(kast / REF_MEDIANO['KAST']),
        'HS':         clip(math.sqrt(hs / REF_MEDIANO['HS'])),
        '+/-':        clip(1 + (kd_rate  / 2)),  
        '+/-_FK_FD':  clip(1 + (fkd_rate / 1)),   
    }

    score = sum(PESOS[m] * scores[m] for m in scores)
    return round(score, 3)


# DF do Mapa com o status do jogador e seu impacto

In [12]:
def normalizar_time(nome):
    tag = {
    'SEN':  'Sentinels',
    'NRG':  'NRG',
    'C9':   'Cloud9',
    'FUR':  'FURIA',
    'LEV':  'LEVIATÁN',
    'EG':   'Evil Geniuses',
    '100T': '100 Thieves',
    'KRU':  'KRÜ Esports',
    'G2':   'G2 Esports',
    'ENVY':  'ENVY',
    'MIBR':  'MIBR',
    'LOUD': 'LOUD',
}
    for sigla, completo in tag.items():
        if nome == sigla or nome == completo:
            return sigla
    return nome

In [13]:
times = {
    'LOUD': ['Darker', 'cauanzin', 'Virtyy', 'pANcada', 'lukxo'],
    'C9':   ['OXY', 'penny', 'v1c', 'Xeppaa', 'Zellsis', 'Demon1'],
    'G2':   ['BABYBAY', 'trent', 'valyn', 'leaf', 'jawgemo'],
    '100T': ['Cryocells', 'bang', 'vora', 'Asuna', 'Timotino'],
    'ENVY': ['Rossy', 'Inspire', 'keznit', 'Eggsterr', 'P0PPIN', 'Demon1'],
    'EG':   ['dgzin', 'bao', 'Okeanos', 'C0M', 'supamen'],
    'KRÜ':  ['benG', 'mta', 'NagZ', 'infiltrator', 'Governor'],
    'FUR':  ['koalanoob', 'artzin', 'nerve', 'eeiu', 'alym'],
    'LEV':  ['PxS', 'Sato', 'kiNgg', 'blowz', 'spike', 'Neon'],
    'NRG':  ['brawk', 'keiko', 'mada', 'Ethan', 'skuba'],
    'MIBR': ['aspas', 'Mazino', 'Verno', 'tex', 'zekken'],
    'SEN':  ['johnqt', 'cortezia', 'Reduxx', 'Kyu', 'N4RRATE'],
}

jogador_time_atual = {
    jogador: sigla
    for sigla, jogadores in times.items()
    for jogador in jogadores
}


mapas = ['Haven', 'Abyss', 'Corrode', 'Bind', 'Pearl', 'Breeze',
         'Split', 'Lotus', 'Icebox', 'Sunset', 'Ascent', 'Fracture']


resultados = []

for mapa in mapas:
    df_mapa = df_merged[df_merged["Mapa"] == mapa].copy()

    df_mapa["Time"]     = df_mapa["Time"].apply(normalizar_time)
    df_mapa["Vencedor"] = df_mapa["Vencedor"].apply(normalizar_time)

    if df_mapa.empty:
        print(f"{mapa} — sem dados")
        continue

    for jogador, df_j in df_mapa.groupby("Jogador"):

        tot_k  = int(df_j["K"].sum())
        tot_d  = int(df_j["D"].sum())
        tot_a  = int(df_j["A"].sum())
        tot_fk = int(df_j["FK"].sum())
        tot_fd = int(df_j["FD"].sum())

        med_r    = df_j["R"].mean()
        med_acs  = df_j["ACS"].mean()
        med_kast = df_j["KAST"].mean()
        med_adr  = df_j["ADR"].mean()
        med_hs   = df_j["HS%"].mean()

        jogos   = len(df_j)
        wins    = (df_j["Time"] == df_j["Vencedor"]).sum()
        winrate = wins / jogos if jogos > 0 else 0

        time_atual_jogador = (
            jogador_time_atual.get(jogador)
            or df_j["Time"].mode()[0]  
        )

        impacto = calcular_impacto(
            r=med_r, acs=med_acs, kast=med_kast, adr=med_adr, hs=med_hs,
            tot_k=tot_k, tot_d=tot_d, tot_fk=tot_fk, tot_fd=tot_fd,
            jogos=jogos
        )

        resultados.append({
            "Jogador":   jogador,
            "Time":      time_atual_jogador,
            "Mapa":      mapa,
            "R":         round(med_r,    3),
            "ACS":       round(med_acs,  1),
            "K":         tot_k,
            "D":         tot_d,
            "A":         tot_a,
            "+/-":       tot_k - tot_d,
            "KAST":      round(med_kast, 1),
            "ADR":       round(med_adr,  1),
            "HS%":       round(med_hs,   1),
            "FK":        tot_fk,
            "FD":        tot_fd,
            "+/-_FK_FD": tot_fk - tot_fd,
            "Jogos":     jogos,
            "Wins":      int(wins),
            "WinRate":   round(winrate, 2),
            "Impacto":   impacto,
        })

    print(f"✅ {mapa:<12} — processado")


df_final = (
    pd.DataFrame(resultados)
    .sort_values(["Mapa", "Impacto"], ascending=[True, False])
    .reset_index(drop=True)
)

print(f"\n📦 {df_final['Mapa'].nunique()} mapas | {len(df_final)} linhas")

df_final.to_csv("df_jogador_por_mapas.csv", index=False)

df_final

✅ Haven        — processado
✅ Abyss        — processado
✅ Corrode      — processado
✅ Bind         — processado
✅ Pearl        — processado
✅ Breeze       — processado
✅ Split        — processado
✅ Lotus        — processado
✅ Icebox       — processado
✅ Sunset       — processado
✅ Ascent       — processado
✅ Fracture     — processado

📦 12 mapas | 887 linhas


,Jogador,Time,Mapa,R,ACS,K,D,A,+/-,KAST,ADR,HS%,FK,FD,+/-_FK_FD,Jogos,Wins,WinRate,Impacto
0,infiltrator,KRÜ,Abyss,1.360,293.5,43,30,6,13,87.5,186.0,32.0,5,2,3,2,0,0.0,0.881
1,lukxo,LOUD,Abyss,1.443,279.7,77,44,8,33,75.7,174.7,34.7,12,10,2,3,0,0.0,0.864
2,Reduxx,SEN,Abyss,1.230,229.0,18,12,5,6,76.0,158.0,58.0,4,1,3,1,1,1.0,0.834
3,koalanoob,FUR,Abyss,1.250,271.3,58,41,18,17,79.3,169.0,28.3,13,9,4,3,3,1.0,0.831
4,heat,FUR,Abyss,1.170,222.0,16,12,1,4,75.0,144.0,59.0,4,0,4,1,0,0.0,0.812
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
882,s0m,NRG,Sunset,0.640,152.0,44,68,19,-24,55.8,104.0,36.5,6,16,-10,4,0,0.0,0.488
883,yay,EG,Sunset,0.660,161.0,9,17,7,-8,55.0,115.0,21.0,6,7,-1,1,0,0.0,0.477
884,tuyz,FUR,Sunset,0.565,147.0,16,30,13,-14,64.0,92.0,34.0,2,6,-4,2,0,0.0,0.475
885,Urango,FUR,Sunset,0.525,119.0,13,29,8,-16,63.5,88.0,37.5,0,4,-4,2,0,0.0,0.457


In [23]:
df_final[df_final["Jogador"] == 'zekken']

,Jogador,Time,Mapa,R,ACS,K,D,A,+/-,KAST,ADR,HS%,FK,FD,+/-_FK_FD,Jogos,Wins,WinRate,Impacto
50,zekken,MIBR,Abyss,0.906,182.6,85,92,39,-7,69.0,125.0,28.7,20,22,-2,7,3,0.43,0.870
98,zekken,MIBR,Ascent,1.012,218.2,70,65,13,5,72.2,139.2,27.8,20,20,0,4,2,0.50,1.115
165,zekken,MIBR,Bind,1.047,200.5,63,55,20,8,75.5,130.5,21.2,13,11,2,4,3,0.75,1.132
241,zekken,MIBR,Breeze,1.297,237.3,46,36,16,10,80.7,150.3,27.0,11,6,5,3,2,0.67,1.273
297,zekken,MIBR,Corrode,1.226,247.3,126,101,40,25,77.4,159.6,29.9,26,21,5,7,5,0.71,1.260
412,zekken,MIBR,Fracture,0.880,160.0,10,13,2,-3,61.0,113.0,25.0,2,3,-1,1,0,0.00,0.799
454,zekken,MIBR,Haven,1.031,205.5,153,143,54,10,73.5,129.1,28.1,29,37,-8,11,5,0.45,1.074
529,zekken,MIBR,Icebox,0.970,212.8,84,78,23,6,70.4,138.6,33.8,20,13,7,5,2,0.40,1.122
594,zekken,MIBR,Lotus,1.127,241.1,192,156,47,36,74.2,153.8,21.5,45,46,-1,11,7,0.64,1.174
671,zekken,MIBR,Pearl,1.125,243.0,195,171,59,24,72.3,154.0,26.4,47,37,10,10,7,0.70,1.204


# Estatistica Geral dos Jogadores

In [5]:
times = {
    'LOUD': ['Darker', 'cauanzin', 'Virtyy', 'pANcada', 'lukxo'],
    'C9':   ['OXY', 'penny', 'v1c', 'Xeppaa', 'Zellsis', 'Demon1'],
    'G2':   ['BABYBAY', 'trent', 'valyn', 'leaf', 'jawgemo'],
    '100T': ['Cryocells', 'bang', 'vora', 'Asuna', 'Timotino'],
    'ENVY': ['Rossy', 'Inspire', 'keznit', 'Eggsterr', 'P0PPIN', 'Demon1'],
    'EG':   ['dgzin', 'bao', 'Okeanos', 'C0M', 'supamen'],
    'KRÜ':  ['benG', 'mta', 'NagZ', 'infiltrator', 'Governor'],
    'FUR':  ['koalanoob', 'artzin', 'nerve', 'eeiu', 'alym'],
    'LEV':  ['PxS', 'Sato', 'kiNgg', 'blowz', 'spike', 'Neon'],
    'NRG':  ['brawk', 'keiko', 'mada', 'Ethan', 'skuba'],
    'MIBR': ['aspas', 'Mazino', 'Verno', 'tex', 'zekken'],
    'SEN':  ['johnqt', 'cortezia', 'Reduxx', 'Kyu', 'N4RRATE'],
}

# função para achar o time
def pegar_time(jogador, df_j):
    for time, players in times.items():
        if jogador in players:
            return time
    return df_j.sort_values("Mapa_Num")["Time"].iloc[-1]


linhas = []

for jogador in df_merged["Jogador"].unique():
    df_j = df_merged[df_merged["Jogador"] == jogador]

    # SOMAS 
    tot_k  = int(df_j["K"].sum())
    tot_d  = int(df_j["D"].sum())
    tot_a  = int(df_j["A"].sum())
    tot_fk = int(df_j["FK"].sum())
    tot_fd = int(df_j["FD"].sum())

    score_time = df_j["Score_Time"].sum()
    score_adv  = df_j["Score_Adv"].sum()
    mapas_time = df_j["Mapas_Time"].sum()
    mapas_adv  = df_j["Mapas_Adv"].sum()

    # MÉDIAS
    med_r    = df_j["R"].mean()
    med_acs  = df_j["ACS"].mean()
    med_kast = df_j["KAST"].mean()
    med_adr  = df_j["ADR"].mean()
    med_hs   = df_j["HS%"].mean()

    # DERIVADOS 
    diff_kd = tot_k - tot_d
    diff_fk_fd = tot_fk - tot_fd

    # TIME
    time_final = pegar_time(jogador, df_j)

    linhas.append({
        "Jogador": jogador,
        "Time": time_final,

        "K": tot_k,
        "D": tot_d,
        "A": tot_a,
        "+/-": diff_kd,

        "FK": tot_fk,
        "FD": tot_fd,
        "+/-_FK_FD": diff_fk_fd,

        "R": med_r,
        "ACS": med_acs,
        "KAST": med_kast,
        "ADR": med_adr,
        "HS%": med_hs,

        "Score_Time": score_time,
        "Score_Adv": score_adv,
        "Mapas_Time": mapas_time,
        "Mapas_Adv": mapas_adv,
    })

df_geral = pd.DataFrame(linhas)

# Formatação
df_geral["R"]    = df_geral["R"].round(3)
df_geral["ACS"]  = df_geral["ACS"].round(1)
df_geral["KAST"] = df_geral["KAST"].round(1)
df_geral["ADR"]  = df_geral["ADR"].round(1)
df_geral["HS%"]  = df_geral["HS%"].round(1)

# Inteiros
for col in ["K", "D", "A", "+/-", "FK", "FD", "+/-_FK_FD",
            "Score_Time", "Score_Adv", "Mapas_Time", "Mapas_Adv"]:
    df_geral[col] = pd.to_numeric(df_geral[col], errors="coerce").fillna(0).astype(int)

# Ordem das colunas
colunas = [
    "Jogador", "Time",
    "R", "ACS", "KAST", "ADR", "HS%",
    "K", "D", "A", "+/-",
    "FK", "FD", "+/-_FK_FD",
    "Score_Time", "Score_Adv",
    "Mapas_Time", "Mapas_Adv"
]

df_geral

,Jogador,Time,K,D,A,+/-,FK,FD,+/-_FK_FD,R,ACS,KAST,ADR,HS%,Score_Time,Score_Adv,Mapas_Time,Mapas_Adv
0,Darker,LOUD,262,288,112,-26,34,34,0,0.932,195.5,70.0,123.0,25.0,148,232,4,26
1,cauanzin,LOUD,1248,1424,512,-176,148,208,-60,0.896,187.7,68.6,124.8,25.1,892,1062,52,110
2,Virtyy,LOUD,648,694,160,-46,144,182,-38,0.885,195.5,66.2,121.6,24.7,400,532,20,58
3,pANcada,LOUD,1264,1312,642,-48,132,108,24,0.990,182.5,72.4,124.4,35.9,892,1062,52,110
4,lukxo,LOUD,1130,950,244,180,204,146,58,1.104,233.8,68.3,152.1,32.6,592,746,32,82
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,Add3r,FUR,56,56,6,0,8,12,-4,0.877,194.7,65.7,125.3,32.7,52,24,6,0
97,stellar,FUR,58,48,22,10,8,12,-4,1.077,204.7,76.3,143.3,19.0,52,24,6,0
98,jakee,FUR,58,50,36,8,2,8,-6,1.173,221.0,82.0,143.0,25.0,52,24,6,0
99,mwzera,FUR,88,110,46,-22,14,14,0,0.873,166.3,66.2,111.2,19.5,80,80,6,6


In [6]:
REF_MEDIANO = {
    'K': 770.0, 'D': 840.0, 'A': 294.0, '+/-': -26.0,
    'FK': 120.0, 'FD': 124.0, '+/-_FK_FD': -4.0,
    'R': 0.949, 'ACS': 190.8, 'KAST': 71.1, 'ADR': 124.8,
    'HS': 27.5, 'Score_Time': 548.0, 'Score_Adv': 630.0,
    'Mapas_Time': 48.0, 'Mapas_Adv': 56.0
}

PESOS = {
    'ACS': 0.106, 'R': 0.104, 'ADR': 0.097, 'HS': 0.095,
    'KAST': 0.093, '+/-': 0.085, 'A': 0.079, 'D': 0.076,
    'K': 0.071, '+/-_FK_FD': 0.056, 'FK': 0.052, 'FD': 0.050,
    'Mapa_Num': 0.034
}

def calcular_impacto(r, acs, kast, adr, hs, tot_k, tot_d, tot_fk, tot_fd, jogos):
    def clip(x, min_v=0.5, max_v=1.5):
        return max(min(x, max_v), min_v)

    kd_rate  = (tot_k - tot_d) / jogos if jogos > 0 else 0
    fkd_rate = (tot_fk - tot_fd) / jogos if jogos > 0 else 0

    scores = {
        'R':         clip(r    / REF_MEDIANO['R']),
        'ACS':       clip(acs  / REF_MEDIANO['ACS']),
        'ADR':       clip(adr  / REF_MEDIANO['ADR']),
        'KAST':      clip(kast / REF_MEDIANO['KAST']),
        'HS':        clip(math.sqrt(hs / REF_MEDIANO['HS'])),
        '+/-':       clip(1 + (kd_rate  / 2)),
        '+/-_FK_FD': clip(1 + (fkd_rate / 1)),
    }

    pesos_usados = {k: v for k, v in PESOS.items() if k in scores}
    score = sum(pesos_usados[m] * scores[m] for m in pesos_usados)
    return round(score, 3)

df_geral['jogos'] = df_geral['Mapas_Time'] + df_geral['Mapas_Adv']

df_geral['Impacto'] = df_geral.apply(lambda row: calcular_impacto(
    r=row['R'],
    acs=row['ACS'],
    kast=row['KAST'],
    adr=row['ADR'],
    hs=row['HS%'],
    tot_k=row['K'],
    tot_d=row['D'],
    tot_fk=row['FK'],
    tot_fd=row['FD'],
    jogos=row['jogos']
), axis=1)

df_geral.drop(columns='jogos', inplace=True)
df_geral.to_csv("df_jogador_geral.csv", index=False)
df_geral

,Jogador,Time,K,D,A,+/-,FK,FD,+/-_FK_FD,R,ACS,KAST,ADR,HS%,Score_Time,Score_Adv,Mapas_Time,Mapas_Adv,Impacto
0,Darker,LOUD,262,288,112,-26,34,34,0,0.932,195.5,70.0,123.0,25.0,148,232,4,26,0.593
1,cauanzin,LOUD,1248,1424,512,-176,148,208,-60,0.896,187.7,68.6,124.8,25.1,892,1062,52,110,0.558
2,Virtyy,LOUD,648,694,160,-46,144,182,-38,0.885,195.5,66.2,121.6,24.7,400,532,20,58,0.565
3,pANcada,LOUD,1264,1312,642,-48,132,108,24,0.990,182.5,72.4,124.4,35.9,892,1062,52,110,0.647
4,lukxo,LOUD,1130,950,244,180,204,146,58,1.104,233.8,68.3,152.1,32.6,592,746,32,82,0.773
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,Add3r,FUR,56,56,6,0,8,12,-4,0.877,194.7,65.7,125.3,32.7,52,24,6,0,0.604
97,stellar,FUR,58,48,22,10,8,12,-4,1.077,204.7,76.3,143.3,19.0,52,24,6,0,0.677
98,jakee,FUR,58,50,36,8,2,8,-6,1.173,221.0,82.0,143.0,25.0,52,24,6,0,0.716
99,mwzera,FUR,88,110,46,-22,14,14,0,0.873,166.3,66.2,111.2,19.5,80,80,6,6,0.540


# Testes até obter algum resultado ultil

In [65]:
df_haven_stats[df_haven_stats["Jogador"] == "bao"]

,Jogador,Time,R,ACS,K,D,A,+/-,KAST,ADR,HS%,FK,FD,+/-_FK_FD,Jogos,Wins,WinRate,Impacto
48,bao,EG,0.845,165.0,24,26,9,-2,75.5,115.5,28.5,2,1,1,2,1,0.5,0.933


In [13]:
cols = ["R", "ACS", "K", "D", "A", "+/-", "ADR", "FK", "FD", "+/-_FK_FD"]
mapas = ['Haven', 'Abyss', 'Corrode', 'Bind', 'Pearl', 'Breeze', 'Split', 'Lotus', 'Icebox', 'Sunset', 'Ascent', 'Fracture']
tag = ['SEN','NRG','C9','FUR','LEV','EG','100T','KRÜ','G2','ENVY','MIBR','LOUD']
        
df_stats2[(df_stats2["Jogador"] == "Darker") & (df_stats2["Mapa"] == "Haven")]

,URL,Mapa_Num,Mapa,Jogador,Time,Agente,R,ACS,K,D,A,+/-,KAST,ADR,HS%,FK,FD,+/-_FK_FD
0,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,Darker,LOUD,Viper,1.12,256,18,16,6,2,70,166,30,2,1,1
70,https://www.vlr.gg/596398/loud-vs-cloud9-vct-2...,0,Haven,Darker,LOUD,Viper,1.12,256,18,16,6,2,70,166,30,2,1,1
640,https://www.vlr.gg/596419/loud-vs-evil-geniuse...,0,Haven,Darker,LOUD,Viper,1.20,288,18,15,4,3,78,167,29,3,2,1


In [8]:
tot_k = int(df_stats2[(df_stats2["Jogador"] == "Darker") & (df_stats2["Mapa"] == "Haven")]["K"].sum())
tot_d = int(df_stats2[(df_stats2["Jogador"] == "Darker") & (df_stats2["Mapa"] == "Haven")]["D"].sum())
tot_a = int(df_stats2[(df_stats2["Jogador"] == "Darker") & (df_stats2["Mapa"] == "Haven")]["A"].sum())
tot_fk = int(df_stats2[(df_stats2["Jogador"] == "Darker") & (df_stats2["Mapa"] == "Haven")]["FK"].sum())
tot_fd = int(df_stats2[(df_stats2["Jogador"] == "Darker") & (df_stats2["Mapa"] == "Haven")]["FD"].sum())

med_r = df_stats2[(df_stats2["Jogador"] == "Darker") & (df_stats2["Mapa"] == "Haven")]["R"].mean()
med_acs = df_stats2[(df_stats2["Jogador"] == "Darker") & (df_stats2["Mapa"] == "Haven")]["ACS"].mean()
med_kast = df_stats2[(df_stats2["Jogador"] == "Darker") & (df_stats2["Mapa"] == "Haven")]["KAST"].mean()
med_adr = df_stats2[(df_stats2["Jogador"] == "Darker") & (df_stats2["Mapa"] == "Haven")]["ADR"].mean()
med_hs = df_stats2[(df_stats2["Jogador"] == "Darker") & (df_stats2["Mapa"] == "Haven")]["HS%"].mean()

dif_kd = tot_k - tot_d
dif_fkd = tot_fk - tot_fd

print(f"k={tot_k} / d={tot_d} / a={tot_a} / fk={tot_fk} / fd={tot_fd}")
print(f"dif_KD={dif_kd} / dif_FKD={dif_fkd}")
print(f"r={med_r} / acs={med_acs} / kast={med_kast} / adr={med_adr}/ hs={med_hs} ")

k=54 / d=47 / a=16 / fk=7 / fd=4
dif_KD=7 / dif_FKD=3
r=1.1466666666666667 / acs=266.6666666666667 / kast=72.66666666666667 / adr=166.33333333333334/ hs=29.666666666666668 


In [27]:
REF_MEDIANO = {
    'R':    0.98,
    'ACS':  192.0,
    'ADR':  126.0,
    'KAST': 73.0,
    'HS':   27.0,
    'KD':   0,
    'FKD':  0.0,
}

FAIXAS = {
    'R':    {'elite': 1.19,  'bom': 0.98,  'mediano': 0.78},
    'ACS':  {'elite': 228.0, 'bom': 192.0, 'mediano': 159.0},
    'ADR':  {'elite': 149.0, 'bom': 126.0, 'mediano': 105.0},
    'KAST': {'elite': 79.0,  'bom': 73.0,  'mediano': 65.0},
    'HS':   {'elite': 33.0,  'bom': 27.0,  'mediano': 21.0},
    'KD':   {'elite': 1,     'bom': 0,     'mediano': -5},
    'FKD':  {'elite': 1.0,   'bom': 0.0,   'mediano': -2.0},
}

PESOS = {
    'R':    0.25,
    'ACS':  0.20,
    'ADR':  0.15,
    'KAST': 0.15,
    'KD':   0.10,
    'HS':   0.05,
    'FKD':  0.10,
}


def classificar(valor, faixa):
    """Retorna 3=elite, 2=bom, 1=mediano, 0=abaixo."""
    if valor >= faixa['elite']:
        return 3
    elif valor >= faixa['bom']:
        return 2
    elif valor >= faixa['mediano']:
        return 1
    return 0


def calcular_impacto(r, acs, kast, adr, hs, tot_k, tot_d, tot_fk, tot_fd):
    dif_kd  = tot_k  - tot_d
    dif_fkd = tot_fk - tot_fd

    # Normaliza cada métrica pelo valor de referência mediano
    # KD e FKD usam diferença absoluta em vez de ratio
    scores = {
        'R':    r    / max(REF_MEDIANO['R'],    0.01),
        'ACS':  acs  / max(REF_MEDIANO['ACS'],  0.01),
        'ADR':  adr  / max(REF_MEDIANO['ADR'],  0.01),
        'KAST': kast / max(REF_MEDIANO['KAST'], 0.01),
        'HS':   hs   / max(REF_MEDIANO['HS'],   0.01),
        'KD':   1 + (dif_kd  / 10),   # shift: 0 = neutro, +1 = excelente
        'FKD':  1 + (dif_fkd / 5),    # shift: 0 = neutro, +1 = excelente
    }

    # Score ponderado final
    score = sum(PESOS[m] * scores[m] for m in PESOS)

    # Classificação por faixa de cada métrica
    print("\n  📊 Detalhamento por métrica:")
    print(f"  {'Métrica':<8} {'Valor':>8}  {'Nível':<10} {'Contribuição':>12}")
    print(f"  {'─'*44}")
    for m, val, faixa_key in [
        ('R',    r,       'R'),
        ('ACS',  acs,     'ACS'),
        ('ADR',  adr,     'ADR'),
        ('KAST', kast,    'KAST'),
        ('HS',   hs,      'HS'),
        ('KD',   dif_kd,  'KD'),
        ('FKD',  dif_fkd, 'FKD'),
    ]:
        nivel_num = classificar(val, FAIXAS[faixa_key])
        nivel_str = ['🔴 Fraco', '🟡 Mediano', '🟢 Bom', '🌟 Elite'][nivel_num]
        contrib   = PESOS[faixa_key] * scores[faixa_key]
        print(f"  {m:<8} {val:>8.2f}  {nivel_str:<12} {contrib:>10.3f}")

    return round(score, 3)


# ─────────────────────────────────────────────
impacto_bao = calcular_impacto(
    r=med_r, acs=med_acs, kast=med_kast, adr=med_adr, hs=med_hs,
    tot_k=tot_k, tot_d=tot_d, tot_fk=tot_fk, tot_fd=tot_fd
)

print(f"\n  ⭐ Score de Impacto: {impacto_bao}")
print(f"  Referência: < 0.90 fraco | 0.90-1.00 mediano | 1.00-1.15 bom | > 1.15 elite")